# Q6. PPO：Advantage 与 Clipping


## Advantage 是什么，为什么比 reward 好

直接用 reward $R$ 作为权重有一个问题：如果所有动作的 reward 都是正数（比如 $+5, +3, +7$），Policy Gradient 会**同时提高所有动作的概率**，训练信号模糊，收敛很慢。

**Advantage 的定义**：

$$A = R - \text{baseline} \qquad (\text{baseline 通常是平均 reward})$$

给定三个动作的 reward $= [+5, +3, +7]$，baseline $= 5$：

1. 计算三个动作的 advantage
2. 解释：advantage 为负的动作，概率应该怎么变？这比直接用 reward 好在哪里？

**验收标准**：能说清楚 advantage 是「比平均水平好多少」，而非绝对好坏。

### 解答

**Step 1：计算 advantage**

| 动作 | reward $R$ | baseline | advantage $A = R - \text{baseline}$ |
|------|-----------|----------|--------------------------------------|
| $a_1$ | +5 | 5 | $0$ |
| $a_2$ | +3 | 5 | $-2$ |
| $a_3$ | +7 | 5 | $+2$ |

**Step 2：advantage 为负时概率怎么变**

Policy Gradient 的更新方向是 $A \cdot \nabla \log \pi$：
- $A > 0$（$a_3$）：梯度为正 → **提高** $a_3$ 的概率（比平均好，多选）
- $A = 0$（$a_1$）：梯度为零 → 概率**不变**（恰好等于平均水平，无信号）
- $A < 0$（$a_2$）：梯度为负 → **降低** $a_2$ 的概率（比平均差，少选）

**Step 3：为什么比直接用 reward 好**

直接用 $R = [+5, +3, +7]$：三个值全为正，模型**同时**被告知提高三个动作的概率，互相竞争，信号混乱，只有靠概率归一化来间接压制差的动作，收敛慢。

用 $A = [0, -2, +2]$：信号变得清晰——**明确告诉模型 $a_2$ 该降、$a_3$ 该升**，训练效率更高。

> Advantage 衡量的是「比平均水平好多少」，而非「绝对好坏」。reward 全为正不代表所有动作都值得鼓励，低于平均的动作同样应该被抑制。

## Probability Ratio 与 Clipping

PPO 的 loss 不直接用 $\log \pi$，而是用新旧 policy 的概率比：

$$r(\theta) = \frac{\pi_{\text{new}}(a)}{\pi_{\text{old}}(a)}, \qquad \text{PPO objective（未 clip）} = r(\theta) \cdot A$$

给定：$\pi_{\text{old}} = 0.3$，$\pi_{\text{new}} = 0.6$，$A = +2$，$\varepsilon = 0.2$

1. 计算 $r(\theta)$ 和 unclipped objective
2. clip 范围是 $[1-\varepsilon,\ 1+\varepsilon] = [0.8,\ 1.2]$，计算 clipped objective
3. 最终 $\text{loss} = -\min(\text{unclipped},\ \text{clipped})$，计算结果
4. 解释：为什么 $r=2.0$ 时要被截断？这防止了什么问题？

**验收标准**：能手算全过程，并说清楚 clip 防止「步子太大」的直觉。

### 手算推导

**Step 1：计算 $r(\theta)$ 和 unclipped objective**

$$r(\theta) = \frac{\pi_{\text{new}}}{\pi_{\text{old}}} = \frac{0.6}{0.3} = 2.0$$

$$\text{unclipped} = r \cdot A = 2.0 \times 2 = \boxed{4.0}$$

**Step 2：clipped objective**

$r = 2.0$ 超出了上限 $1.2$，被截断：

$$\text{clip}(2.0,\ 0.8,\ 1.2) = 1.2$$

$$\text{clipped} = 1.2 \times 2 = \boxed{2.4}$$

**Step 3：最终 loss**

$$\text{loss} = -\min(\text{unclipped},\ \text{clipped}) = -\min(4.0,\ 2.4) = \boxed{-2.4}$$

**Step 4：为什么要截断 $r=2.0$？**

$r=2.0$ 说明新策略对这个动作的概率是旧策略的 2 倍——**单步更新幅度过大**。

此时如果不截断，梯度会继续推动 $\pi_{\text{new}}$ 进一步增大，但我们用的训练数据是旧策略采集的，当新旧策略差距太大时，这批数据对新策略来说已经不准确了（importance sampling 的估计方差爆炸）。

Clipping 把 $r$ 限制在 $[0.8, 1.2]$，等于强制新策略「别离旧策略太远」，每次只走一小步，保证训练数据还有参考价值。

取 $\min$ 的作用：选择更保守的那个目标值，当 $r$ 超出范围时，clipped 项的梯度为 0，**自动停止在该方向上继续更新**。

In [1]:
import torch

pi_old = torch.tensor(0.3)
pi_new = torch.tensor(0.6)
A      = torch.tensor(2.0)
eps    = 0.2

# Step 1: probability ratio + unclipped objective
r = pi_new / pi_old
unclipped = r * A
print(f"r = {r.item():.4f}  (expected: 2.0)")
print(f"unclipped objective = {unclipped.item():.4f}  (expected: 4.0)")

# Step 2: clipped objective
r_clipped = torch.clamp(r, 1 - eps, 1 + eps)
clipped   = r_clipped * A
print(f"r_clipped = {r_clipped.item():.4f}  (expected: 1.2)")
print(f"clipped objective = {clipped.item():.4f}  (expected: 2.4)")

# Step 3: final PPO loss
loss = -torch.min(unclipped, clipped)
print(f"PPO loss = {loss.item():.4f}  (expected: -2.4)")

r = 2.0000  (expected: 2.0)
unclipped objective = 4.0000  (expected: 4.0)
r_clipped = 1.2000  (expected: 1.2)
clipped objective = 2.4000  (expected: 2.4)
PPO loss = -2.4000  (expected: -2.4)


## 从 $J(\theta)$ 到概率比：中间发生了什么

$J(\theta) = \mathbb{E}[R \cdot \log \pi_\theta]$ 是我们想最大化的目标，梯度是 $\mathbb{E}[A \cdot \nabla \log \pi_\theta]$。

普通 Policy Gradient 每次更新完就扔掉数据，因为数据是旧策略采的，更新一步后就不准了。

**PPO 想做的事**：用同一批数据更新多步（更高效），但更新多步后新旧策略越来越不一样，数据就越来越不准。

**怎么解决**：用 Importance Sampling。如果数据是用 $\pi_{\text{old}}$ 采集的，但想估计 $\pi_{\text{new}}$ 下的期望，可以做权重修正：

$$\mathbb{E}_{\pi_{\text{new}}}[A] = \mathbb{E}_{\pi_{\text{old}}}\left[\frac{\pi_{\text{new}}}{\pi_{\text{old}}} \cdot A\right] = \mathbb{E}_{\pi_{\text{old}}}\left[r(\theta) \cdot A\right]$$

这就是概率比 $r(\theta) = \pi_{\text{new}} / \pi_{\text{old}}$ 的来源——**它是 Importance Sampling 的修正权重**，把「旧数据」修正成对新策略的估计。

所以整条演化链是：

```
J(θ) 最大化期望 reward
    ↓ 想多用几次同一批数据
Importance Sampling：乘上 r(θ) = π_new / π_old 做修正
    ↓ r(θ) 可能太大，修正不准
加 Clipping：限制 r(θ) ∈ [1-ε, 1+ε]
    ↓
PPO loss = -min(r·A, clip(r, 1-ε, 1+ε)·A)
```

目标从来没变过，还是在最大化 $J(\theta)$，只是换了一种更高效、更稳定的方式去估计和优化它。

## 为什么要加负号

- $J(\theta)$ 是我们想**最大化**的目标（reward 越高越好）
- PyTorch 的优化器做的是**梯度下降**，只会最小化

所以把目标取负号，最大化 $J(\theta)$ 就变成最小化 $-J(\theta)$：

$$\text{loss} = -J(\theta) = -\mathbb{E}[r(\theta) \cdot A]$$

加上 clipping 后：

$$\text{loss} = -\min\left(r(\theta) \cdot A,\ \text{clip}(r(\theta), 1-\varepsilon, 1+\varepsilon) \cdot A\right)$$

这就是为什么 PPO 的 loss 前面有个负号。**负号不是在惩罚什么，只是把「最大化 reward」翻译成了优化器能理解的语言。**